# 05 — High-Accuracy Training Run + Honest Dual-Metric Evaluation

Notebooks 02/03 used a tiny writer subset (5-8 writers, a handful of epochs) purely
to prove the pipeline trains on real data quickly. This notebook trains both
branches on a **larger real writer subset with more epochs**, and — this is the
important part — evaluates them with the same metric split the signature-
verification literature uses, instead of one blended number:

- **Skilled-forgery accuracy**: genuine vs. a deliberate forgery of *that specific
  writer's* signature (CEDAR's `full_forg`, MOBISIG's `SIGN_FOR_*`). This is the
  hard, realistic threat model and the number that matters for a real deployment.
- **Random-forgery accuracy**: genuine vs. a genuine signature *from a different
  writer* (an "impostor" who isn't even trying to forge anything). This is a much
  easier discrimination task — different people's handwriting is usually very
  distinct — and routinely scores much higher. Reporting this alone, unlabeled,
  is the classic way signature-verification accuracy claims get inflated.

**No number below is fabricated or hand-set.** Both come directly from the
`verification_report` / `equal_error_rate` code in `sigverify.utils.metrics` run
against this notebook's own held-out, writer-disjoint validation split. If you
re-run this notebook you will get very similar but not bit-identical numbers
(dataset subsampling and training are seeded, but PyTorch's CPU kernels aren't
100% bitwise-deterministic across runs).

**On "99.5% accuracy"**: skilled-forgery detection this good would exceed most
published results on CEDAR-scale data (state-of-the-art deep Siamese approaches
typically report skilled-forgery EER in the ~3-15% range, i.e. ~85-97% accuracy,
even with full datasets, full resolution, and GPU-scale training). This notebook
runs on a CPU-only, memory-constrained host with a fraction of CEDAR's 55 writers
and a lightweight backbone — treat its skilled-forgery number as a proof-of-concept
data point, not a production claim. `configs/default.yaml` + `scripts/train_static.py`
on the full dataset with a GPU is the path to a number you'd cite externally.


In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "src"))

import random
import time
import numpy as np
import torch
import cv2
import matplotlib.pyplot as plt

from sigverify.data.datasets import load_manifest, split_writers
from sigverify.preprocessing.image_preprocess import preprocess_signature_image
from sigverify.models.static_branch import SiameseCNN
from sigverify.models.losses import CombinedEmbeddingLoss
from sigverify.utils.metrics import verification_report
from sigverify.utils.seed import set_seed, get_device

set_seed(42)
device = get_device("cpu")
ARTIFACTS = REPO_ROOT / "notebooks/artifacts"
ARTIFACTS.mkdir(parents=True, exist_ok=True)
print("Device:", device)


## Rebuild the real CEDAR manifest with more writers, and cache them once

In [ ]:
import sys as _sys
_sys.path.insert(0, str(REPO_ROOT / "scripts"))
from prepare_real_datasets import build_cedar_manifest

NUM_WRITERS = 25   # of the 55 total CEDAR writers on disk
TARGET_SIZE = (64, 64)
# Preprocessing (denoising) is the expensive part -- and on this host, mostly
# from first-time disk/antivirus overhead on each newly-touched file, not raw
# compute (1200 distinct images took ~33 minutes to cache once). Persist the
# result to disk so re-running this notebook (e.g. after a hyperparameter
# change below) doesn't re-pay that cost.
CACHE_PATH = ARTIFACTS / f"cedar_cache_{NUM_WRITERS}w_{TARGET_SIZE[0]}px.pt"

static_manifest_path = build_cedar_manifest(max_writers=NUM_WRITERS)
static_records = load_manifest(static_manifest_path)
all_writers = sorted({r["writer_id"] for r in static_records}, key=lambda w: int(w.rsplit("_", 1)[1]))
print(f"Using {len(all_writers)} writers, {len(static_records)} samples")

if CACHE_PATH.exists():
    t0 = time.time()
    cache = torch.load(CACHE_PATH, weights_only=True)
    print(f"Loaded {len(cache)} preprocessed images from disk cache in {time.time() - t0:.1f}s ({CACHE_PATH.name})")
else:
    t0 = time.time()
    cache = {}
    for rec in static_records:
        raw = cv2.imread(rec["path"], cv2.IMREAD_UNCHANGED)
        processed = preprocess_signature_image(raw, target_size=TARGET_SIZE)
        cache[rec["path"]] = torch.from_numpy(processed).unsqueeze(0)
    print(f"Cached {len(cache)} preprocessed images in {time.time() - t0:.1f}s")
    torch.save(cache, CACHE_PATH)
    print(f"Persisted disk cache to {CACHE_PATH} for future re-runs")


## Writer-disjoint split + cache-backed triplet sampler (same design as notebook 02)

In [ ]:
train_writers, val_writers = split_writers(static_manifest_path, val_fraction=0.24, seed=42)
print(f"Train writers: {len(train_writers)} | Val writers: {len(val_writers)}")


class CachedTripletSampler:
    def __init__(self, records, writer_ids, cache, impostor_ratio=0.5, seed=0):
        self.cache = cache
        self.impostor_ratio = impostor_ratio
        self.rng = random.Random(seed)
        records = [r for r in records if r["writer_id"] in writer_ids]
        self.by_genuine, self.by_forged = {}, {}
        for r in records:
            bucket = self.by_genuine if r["label"] == "genuine" else self.by_forged
            bucket.setdefault(r["writer_id"], []).append(r)
        self.anchors = [r for w, rs in self.by_genuine.items() if len(rs) >= 2 for r in rs]
        self.writers = list(self.by_genuine.keys())

    def __len__(self):
        return len(self.anchors)

    def batches(self, batch_size, shuffle=True):
        order = list(range(len(self.anchors)))
        if shuffle:
            self.rng.shuffle(order)
        for i in range(0, len(order), batch_size):
            idx_batch = order[i : i + batch_size]
            a, p, n = [], [], []
            for idx in idx_batch:
                anchor_rec = self.anchors[idx]
                writer = anchor_rec["writer_id"]
                pool = [r for r in self.by_genuine[writer] if r["path"] != anchor_rec["path"]]
                positive_rec = self.rng.choice(pool) if pool else anchor_rec
                if self.by_forged.get(writer) and self.rng.random() > self.impostor_ratio:
                    negative_rec = self.rng.choice(self.by_forged[writer])
                else:
                    other = self.rng.choice([w for w in self.writers if w != writer] or [writer])
                    negative_rec = self.rng.choice(self.by_genuine[other])
                a.append(self.cache[anchor_rec["path"]])
                p.append(self.cache[positive_rec["path"]])
                n.append(self.cache[negative_rec["path"]])
            yield torch.stack(a), torch.stack(p), torch.stack(n)

    def dual_eval_batches(self, batch_size):
        """Same anchor/positive pairing, but yields the negative from *only*
        skilled forgeries and *only* random impostors separately, instead of the
        50/50 training mix -- this is what makes the dual-metric evaluation honest."""
        skilled_idx = [i for i, r in enumerate(self.anchors) if self.by_forged.get(r["writer_id"])]
        for i in range(0, len(skilled_idx), batch_size):
            idx_batch = skilled_idx[i : i + batch_size]
            a, p, n = [], [], []
            for idx in idx_batch:
                anchor_rec = self.anchors[idx]
                writer = anchor_rec["writer_id"]
                pool = [r for r in self.by_genuine[writer] if r["path"] != anchor_rec["path"]]
                positive_rec = self.rng.choice(pool) if pool else anchor_rec
                negative_rec = self.rng.choice(self.by_forged[writer])
                a.append(self.cache[anchor_rec["path"]]); p.append(self.cache[positive_rec["path"]]); n.append(self.cache[negative_rec["path"]])
            yield torch.stack(a), torch.stack(p), torch.stack(n), "skilled"

        for i in range(0, len(self.anchors), batch_size):
            idx_batch = list(range(i, min(i + batch_size, len(self.anchors))))
            a, p, n = [], [], []
            for idx in idx_batch:
                anchor_rec = self.anchors[idx]
                writer = anchor_rec["writer_id"]
                pool = [r for r in self.by_genuine[writer] if r["path"] != anchor_rec["path"]]
                positive_rec = self.rng.choice(pool) if pool else anchor_rec
                other = self.rng.choice([w for w in self.writers if w != writer] or [writer])
                negative_rec = self.rng.choice(self.by_genuine[other])
                a.append(self.cache[anchor_rec["path"]]); p.append(self.cache[positive_rec["path"]]); n.append(self.cache[negative_rec["path"]])
            yield torch.stack(a), torch.stack(p), torch.stack(n), "random"


train_sampler = CachedTripletSampler(static_records, train_writers, cache, seed=42)
val_sampler = CachedTripletSampler(static_records, val_writers, cache, seed=43)
print(f"Train anchors: {len(train_sampler)} | Val anchors: {len(val_sampler)}")


## Train (more writers, more epochs than notebook 02)

In [ ]:
BACKBONE = "mobilenet_v3_large"
EPOCHS = 18
BATCH_SIZE = 16

model = SiameseCNN(backbone=BACKBONE, embedding_dim=128, pretrained=True).to(device)
criterion = CombinedEmbeddingLoss(contrastive_margin=1.0, triplet_margin=0.3)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)


@torch.no_grad()
def evaluate_mixed(sampler, batch_size):
    model.eval()
    genuine, forged = [], []
    for a, p, n in sampler.batches(batch_size, shuffle=False):
        a, p, n = a.to(device), p.to(device), n.to(device)
        e_a, e_p = model(a, p)
        e_n = model.embed(n)
        genuine.append(model.similarity(e_a, e_p).cpu().numpy())
        forged.append(model.similarity(e_a, e_n).cpu().numpy())
    genuine = (np.concatenate(genuine) + 1) / 2
    forged = (np.concatenate(forged) + 1) / 2
    return verification_report(genuine, forged)


import copy

history = {"train_loss": [], "val_eer": [], "val_auc": []}
best_eer = float("inf")
best_state = None
best_epoch = 0
patience, patience_left = 6, 6
t_start = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss, num_batches = 0.0, 0
    for a, p, n in train_sampler.batches(BATCH_SIZE, shuffle=True):
        a, p, n = a.to(device), p.to(device), n.to(device)
        optimizer.zero_grad()
        e_a, e_p, e_n = model.embed(a), model.embed(p), model.embed(n)
        loss = criterion(e_a, e_p, e_n)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        num_batches += 1
    scheduler.step()

    metrics = evaluate_mixed(val_sampler, BATCH_SIZE)
    history["train_loss"].append(running_loss / max(1, num_batches))
    history["val_eer"].append(metrics["eer"])
    history["val_auc"].append(metrics["roc_auc"])
    improved = metrics["eer"] < best_eer
    if improved:
        best_eer, best_epoch = metrics["eer"], epoch
        best_state = copy.deepcopy(model.state_dict())
        patience_left = patience
    else:
        patience_left -= 1
    print(f"epoch {epoch:2d}/{EPOCHS} | train_loss={history['train_loss'][-1]:.4f} "
          f"| val_eer(mixed)={metrics['eer']:.4f} | val_auc(mixed)={metrics['roc_auc']:.4f}"
          f"{'  <- best so far' if improved else ''}")
    if patience_left <= 0:
        print(f"Early stopping at epoch {epoch} (no improvement for {patience} epochs)")
        break

print(f"\nTotal training time: {time.time() - t_start:.1f}s")
print(f"Best epoch: {best_epoch} (val_eer(mixed)={best_eer:.4f}) -- restoring those weights for evaluation below")
model.load_state_dict(best_state)  # evaluate/save the BEST checkpoint, not whichever epoch happened to run last
torch.save(model.state_dict(), ARTIFACTS / "static_branch_cedar_highacc.pt")
print("Saved:", ARTIFACTS / "static_branch_cedar_highacc.pt")


In [ ]:
ran_epochs = range(1, len(history["train_loss"]) + 1)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(ran_epochs, history["train_loss"], marker="o")
axes[0].set_title("Training loss")
axes[0].set_xlabel("epoch")
axes[1].plot(ran_epochs, history["val_eer"], marker="o", label="EER, mixed neg. (lower=better)")
axes[1].plot(ran_epochs, history["val_auc"], marker="s", label="ROC-AUC, mixed neg. (higher=better)")
axes[1].axvline(best_epoch, color="green", linestyle="--", alpha=0.6, label=f"best epoch ({best_epoch})")
axes[1].set_title("Validation (held-out writers, 50/50 skilled+random negatives -- matches training distribution)")
axes[1].set_xlabel("epoch")
axes[1].legend()
plt.tight_layout()
plt.show()


## The honest split: skilled-forgery accuracy vs. random-forgery accuracy

In [ ]:
@torch.no_grad()
def evaluate_split(sampler, batch_size):
    model.eval()
    scores = {"skilled": {"genuine": [], "forged": []}, "random": {"genuine": [], "forged": []}}
    for a, p, n, kind in sampler.dual_eval_batches(batch_size):
        a, p, n = a.to(device), p.to(device), n.to(device)
        e_a, e_p = model(a, p)
        e_n = model.embed(n)
        scores[kind]["genuine"].append(model.similarity(e_a, e_p).cpu().numpy())
        scores[kind]["forged"].append(model.similarity(e_a, e_n).cpu().numpy())

    reports = {}
    for kind in ("skilled", "random"):
        genuine = (np.concatenate(scores[kind]["genuine"]) + 1) / 2
        forged = (np.concatenate(scores[kind]["forged"]) + 1) / 2
        reports[kind] = verification_report(genuine, forged)
    return reports

reports = evaluate_split(val_sampler, BATCH_SIZE)

print("=" * 70)
print(f"{'Metric':<28} {'Skilled forgery (hard)':>20} {'Random forgery (easy)':>20}")
print("=" * 70)
for key, label in [("eer", "EER"), ("roc_auc", "ROC-AUC"), ("accuracy_at_eer_threshold", "Accuracy @ EER threshold")]:
    print(f"{label:<28} {reports['skilled'][key]:>20.4f} {reports['random'][key]:>20.4f}")
print("=" * 70)
print(f"\nHeld-out writers: {sorted(val_writers)}")
print(f"Held-out skilled-forgery test pairs: {len(list(val_sampler.dual_eval_batches(BATCH_SIZE))[0][0]) * ((len(val_sampler.anchors) // BATCH_SIZE) or 1)} (approx)")


## Reading these numbers honestly

- **Random-forgery accuracy** is the "genuine vs. a completely different person's
  signature" task. Real, unmodified output of `verification_report` above — expect
  this to be high, often >99%, because different people's handwriting is usually
  very distinct even to a lightly-trained model. This is the number that's easy to
  quote and easy to mis-quote as "the" accuracy.
- **Skilled-forgery accuracy** is the "genuine vs. someone who practiced forging
  this exact signature" task — the number that actually matters for a forensic
  verification claim, and the one every serious paper on this topic reports
  separately because it's so much harder. Whatever value printed above for this
  notebook's small held-out writer subset is the real number: don't round it up,
  and don't quote the random-forgery number in its place.

For a number worth citing externally: train `scripts/train_static.py` on the full
55-writer CEDAR set (or a larger benchmark like GPDS-960) with `configs/default.yaml`
(ResNet50, 224x224, up to 50 epochs with early stopping) on a GPU, and report both
splits the same way this notebook does.
